# 06 - Magnitude Analysis

This notebook compares business measures across key dimensions.

Focus areas:
- Customer distribution by country and gender
- Product distribution by category
- Revenue by product category, country, and customer
- Sales contribution percentages
- Average order value across business groups

In [0]:
%sql
/*
Customer Distribution by Country

Purpose:
    Measure how customers are distributed across countries.
    This gives a geographic view of the customer base.
*/

SELECT
    COALESCE(NULLIF(country, 'n/a'), 'not defined') AS country,
    COUNT(customer_key) AS total_customers,
    ROUND(
        COUNT(customer_key) * 100.0 / SUM(COUNT(customer_key)) OVER (),
        2
    ) AS customer_percentage
FROM datawarehouseanalytics_gold.dim_customers
GROUP BY country
ORDER BY total_customers DESC;

In [0]:
%sql
/*
Customer Distribution by Gender and Country
Purpose:
    Analyze customer distribution by both geography and gender.
    This provides a more detailed demographic breakdown than country alone.
*/

SELECT
    COALESCE(NULLIF(country, 'n/a'), 'not defined') AS country,
    COALESCE(NULLIF(gender, 'n/a'), 'not defined') AS gender,
    COUNT(customer_key) AS total_customers,
    ROUND(
        COUNT(customer_key) * 100.0 
        / SUM(COUNT(customer_key)) OVER (
            PARTITION BY COALESCE(NULLIF(country, 'n/a'), 'not defined')
        ),
        2
    ) AS percentage_within_country
FROM datawarehouseanalytics_gold.dim_customers
GROUP BY 
    COALESCE(NULLIF(country, 'n/a'), 'not defined'),
    COALESCE(NULLIF(gender, 'n/a'), 'not defined')
ORDER BY 
    country, 
    total_customers DESC;



In [0]:
%sql
/*
Product Distribution by Category

Purpose:
    Compare product count and average cost across product categories.
    Missing product categories are grouped as Unknown for clearer reporting.
*/

SELECT
    COALESCE(category, 'Unknown') AS category,
    COUNT(product_key) AS total_products,
    ROUND(AVG(cost), 2) AS avg_cost,
    MIN(cost) AS min_cost,
    MAX(cost) AS max_cost
FROM datawarehouseanalytics_gold.dim_products
GROUP BY COALESCE(category, 'Unknown')
ORDER BY total_products DESC;

In [0]:
%sql
/*
Revenue by Product Category
Purpose:
    Measure total revenue, quantity sold, and order volume by product category.
    This helps identify which product groups contribute most to sales.
*/

SELECT
    COALESCE(p.category, 'Unknown') AS category,
    COUNT(DISTINCT f.order_number) AS total_orders,
    SUM(f.quantity) AS total_quantity,
    SUM(f.sales_amount) AS total_revenue,
    ROUND(
        SUM(f.sales_amount) * 100.0 / SUM(SUM(f.sales_amount)) OVER (),
        2
    ) AS revenue_percentage
FROM datawarehouseanalytics_gold.fact_sales f
LEFT JOIN datawarehouseanalytics_gold.dim_products p
    ON f.product_key = p.product_key
GROUP BY COALESCE(p.category, 'Unknown')
ORDER BY total_revenue DESC;

In [0]:
%sql
/*
Category Revenue Contribution Within Each Country

Purpose:
    Analyze which product categories contribute most to revenue within each
    country. The percentage is calculated within each country.
*/

SELECT
    c.country,
    COALESCE(p.category, 'Unknown') AS category,
    SUM(f.sales_amount) AS total_revenue,
    ROUND(
        SUM(f.sales_amount) * 100.0 
        / SUM(SUM(f.sales_amount)) OVER (PARTITION BY c.country),
        2
    ) AS revenue_percentage_within_country
FROM datawarehouseanalytics_gold.fact_sales f
LEFT JOIN datawarehouseanalytics_gold.dim_customers c
    ON f.customer_key = c.customer_key
LEFT JOIN datawarehouseanalytics_gold.dim_products p
    ON f.product_key = p.product_key
GROUP BY c.country, COALESCE(p.category, 'Unknown')
ORDER BY c.country, total_revenue DESC;